# Exam 01 — sLM General Mastery

**Scope:** Courses 0–4 + production projects  
**Format:** 20 MCQ (20 pts) · 15 Short Answer (30 pts) · 10 Coding (30 pts) · 5 Case Studies (20 pts) = **100 pts**  
**Rules:** No external lookups. All code cells run on CPU with built-in mocks.

---

In [ ]:
# ── Mock setup — runs CPU-only, zero internet, zero GPU ──────────────────────
import sys, types, unittest.mock as mock, numpy as np, json, re, tempfile, os
from pathlib import Path

# Stub heavy packages so imports never touch the network
for pkg in ["torch","transformers","peft","bitsandbytes","datasets",
            "sentence_transformers","faiss","chromadb","langchain",
            "langchain_core","langchain_community","langchain_huggingface",
            "langgraph","langsmith","ragas","instructor","tiktoken"]:
    if pkg not in sys.modules:
        sys.modules[pkg] = types.ModuleType(pkg)

# Tiny numpy-backed embedding helper (no FAISS needed)
def cosine(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))

class FakeEmbed:
    """Deterministic embedder: word-count bag, dim=16."""
    dim = 16
    def embed(self, text):
        v = np.zeros(self.dim)
        for i,w in enumerate(text.lower().split()): v[i%self.dim] += 1
        n = np.linalg.norm(v); return (v/n if n else v).tolist()

class FakeLLM:
    """Always returns the canned reply."""
    def __init__(self, reply="mocked LLM response"): self.reply = reply
    def invoke(self, prompt): return self.reply
    def stream(self, prompt):
        for tok in self.reply.split(): yield tok+" "

class FakeVectorStore:
    """In-memory dot-product store."""
    def __init__(self):
        self._docs=[]; self._vecs=[]; self._emb=FakeEmbed()
    def add(self, texts):
        for t in texts:
            self._docs.append(t); self._vecs.append(self._emb.embed(t))
    def search(self, query, k=3):
        q = self._emb.embed(query)
        scores = [cosine(q,v) for v in self._vecs]
        top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [self._docs[i] for i in top]

print("Mock setup complete — all cells runnable without GPU or internet.")


## Part I — Multiple Choice (20 × 1 pt)

For each question assign your answer to the variable, e.g. `q1 = 'B'`.

### Q1. LCEL pipe operator `|`

What does `prompt | llm | parser` evaluate to in LangChain ≥ 0.3?

- **A**: Three separate objects stored in a tuple
- **B**: A `Runnable` chain where each component's output feeds the next
- **C**: A Python generator that lazily calls each step
- **D**: Syntax sugar for `prompt.invoke(llm.invoke(parser.invoke(...)))`

In [ ]:
q1 = ""  # your answer: A / B / C / D

### Q2. LoRA trainable params

A frozen weight `W` is shape (4096, 4096). LoRA rank r=8. How many trainable params does one adapter add?

- **A**: 4096×4096 = 16,777,216
- **B**: 2×4096×8 = 65,536
- **C**: 4096×8 = 32,768
- **D**: 8×8 = 64

In [ ]:
q2 = ""  # your answer: A / B / C / D

### Q3. QLoRA memory

QLoRA loads the base model in NF4. What is the primary memory benefit vs full BF16?

- **A**: Activations are stored in INT4
- **B**: Base weights use ~4× less VRAM; adapters train in BF16
- **C**: Gradient checkpointing is forced on
- **D**: The optimizer state is discarded

In [ ]:
q3 = ""  # your answer: A / B / C / D

### Q4. EWC penalty

In Elastic Weight Consolidation, the Fisher information matrix diagonal F_i measures:

- **A**: How much parameter i contributed to the current task loss
- **B**: The expected squared gradient of the log-likelihood w.r.t. parameter i on Task A
- **C**: The L2 distance between old and new parameters
- **D**: The cosine similarity between task A and task B gradients

In [ ]:
q4 = ""  # your answer: A / B / C / D

### Q5. FastAPI lifespan

In `learning_slm_claude/courses/projects/01_smolsearch/app/main.py` startup seeding uses `@asynccontextmanager async def lifespan(app)`. What replaced?

- **A**: `@app.on_event('startup')`
- **B**: `@app.middleware('http')`
- **C**: `@app.route('/init')`
- **D**: `@app.exception_handler(500)`

In [ ]:
q5 = ""  # your answer: A / B / C / D

### Q6. parents[N] Docker

`Path(__file__).resolve().parents[4]` crashes in Docker with `WORKDIR /app`, file at `/app/app/main.py`. Why?

- **A**: Docker forbids Path operations
- **B**: The path only has 3 ancestors so index 4 raises IndexError
- **C**: Docker remaps symlinks
- **D**: `__file__` is `None` inside containers

In [ ]:
q6 = ""  # your answer: A / B / C / D

### Q7. DPO vs RLHF

DPO eliminates which component required by standard RLHF?

- **A**: The frozen reference model
- **B**: Human preference labels
- **C**: A separately trained reward model
- **D**: The KL-divergence penalty

In [ ]:
q7 = ""  # your answer: A / B / C / D

### Q8. return_full_text

Without `return_full_text=False` in `HuggingFacePipeline`, the `/answer` endpoint returns:

- **A**: Only the generated tokens
- **B**: The full prompt concatenated with the generated tokens
- **C**: An empty string
- **D**: JSON with a `prompt` key

In [ ]:
q8 = ""  # your answer: A / B / C / D

### Q9. Cloud Run HF_HOME

GCP Cloud Run filesystem is read-only except `/tmp`. The Dockerfile sets `HF_HOME=/tmp/.cache/huggingface`. Without this, model download would:

- **A**: Succeed silently
- **B**: Write to `~/.cache` which is read-only and raise PermissionError
- **C**: Fall back to S3
- **D**: Use in-memory caching automatically

In [ ]:
q9 = ""  # your answer: A / B / C / D

### Q10. Semantic cache hit

A semantic cache stores embeddings of past queries. A new query hits the cache when:

- **A**: The query string is identical (exact match)
- **B**: Cosine similarity between new and stored embedding exceeds a threshold (~0.92)
- **C**: The query length matches
- **D**: The same user issues the request

In [ ]:
q10 = ""  # your answer: A / B / C / D

### Q11. ReAct termination

A ReAct agent loop in `03_agentflow/src/graph.py` terminates when:

- **A**: The LLM output is empty
- **B**: `Final Answer:` is parsed from the LLM output or `max_steps` is reached
- **C**: All tools have been called once
- **D**: The task string ends with a period

In [ ]:
q11 = ""  # your answer: A / B / C / D

### Q12. FAISS vs ChromaDB

FAISS (faiss-cpu) differs from ChromaDB in that FAISS:

- **A**: Supports persistent storage by default
- **B**: Is purely in-memory with no built-in persistence; must be serialised manually
- **C**: Requires a running server process
- **D**: Only supports exact nearest-neighbour search

In [ ]:
q12 = ""  # your answer: A / B / C / D

### Q13. LangGraph vs chain

LangGraph `StateGraph` enables something LCEL chains cannot do natively:

- **A**: Streaming token output
- **B**: Cycles and conditional branching with shared mutable state
- **C**: Parallel map over a list
- **D**: Structured JSON output

In [ ]:
q13 = ""  # your answer: A / B / C / D

### Q14. CI branch strategy

In `.github/workflows/ci.yml` tests run on `feat/**` and `main`. Deploy workflows trigger only on `release`. Why separate branches?

- **A**: GitHub requires it
- **B**: Prevents accidental production deploys from feature branches; `release` is the explicit gate
- **C**: CI is slower on `main`
- **D**: Docker build only works on `release`

In [ ]:
q14 = ""  # your answer: A / B / C / D

### Q15. MMR retrieval

Maximal Marginal Relevance (MMR) retrieval differs from pure similarity search by:

- **A**: Using BM25 scores instead of embeddings
- **B**: Balancing relevance to the query with diversity among returned documents
- **C**: Re-ranking with a cross-encoder
- **D**: Fetching more documents then truncating

In [ ]:
q15 = ""  # your answer: A / B / C / D

### Q16. BPE merges

Byte-Pair Encoding vocabulary size is determined by:

- **A**: The number of unique characters in the corpus
- **B**: The number of merge operations (a hyperparameter)
- **C**: The corpus size in tokens
- **D**: The model hidden dimension

In [ ]:
q16 = ""  # your answer: A / B / C / D

### Q17. GCP IAM deployer

The GitHub Actions deployer service account needs three IAM roles. Which set is correct?

- **A**: `run.admin`, `artifactregistry.writer`, `iam.serviceAccountUser`
- **B**: `run.developer`, `storage.admin`, `iam.admin`
- **C**: `run.viewer`, `artifactregistry.reader`, `iam.serviceAccountUser`
- **D**: `owner`, `editor`, `viewer`

In [ ]:
q17 = ""  # your answer: A / B / C / D

### Q18. PSI drift

Population Stability Index (PSI) in `course3/…/drift.py` measures:

- **A**: Gradient norms during training
- **B**: Distribution shift between a reference and current feature distribution
- **C**: Memory usage over time
- **D**: Latency percentile change

In [ ]:
q18 = ""  # your answer: A / B / C / D

### Q19. git SHA tagging

Docker images are tagged with `${{ github.sha }}` in CI. The main reason is:

- **A**: Shorter tags load faster
- **B**: Each commit produces a unique, traceable image; rollback is trivial
- **C**: SHA tags bypass registry authentication
- **D**: GitHub requires SHA tags

In [ ]:
q19 = ""  # your answer: A / B / C / D

### Q20. bind_tools error

`create_tool_calling_agent(llm, tools, prompt)` raises `ValueError: bind_tools() not implemented`. The root cause is:

- **A**: The prompt template is missing `{agent_scratchpad}`
- **B**: `HuggingFacePipeline` is text-generation only and does not implement `bind_tools()`
- **C**: LangChain 0.3 removed `create_tool_calling_agent`
- **D**: Tools must be registered before the LLM is created

In [ ]:
q20 = ""  # your answer: A / B / C / D

### Part I — Auto-grader
Run the cell below to see your Part I score.

In [ ]:
ANSWER_KEY_P1 = {"q1": "B", "q2": "B", "q3": "B", "q4": "B", "q5": "A", "q6": "B", "q7": "C", "q8": "B", "q9": "B", "q10": "B", "q11": "B", "q12": "B", "q13": "B", "q14": "B", "q15": "B", "q16": "B", "q17": "A", "q18": "B", "q19": "B", "q20": "B"}
answers_p1 = {k: eval(k) for k in ANSWER_KEY_P1}
correct = sum(1 for k,v in ANSWER_KEY_P1.items() if answers_p1.get(k)==v)
total = len(ANSWER_KEY_P1)
print(f'Part I Score: {correct}/{total}')
for k,v in ANSWER_KEY_P1.items():
    your = answers_p1.get(k,'—')
    mark = '✓' if your==v else f'✗ (correct: {v})'
    print(f'  {k}: {your}  {mark}')

---
## Part II — Short Answer (15 × 2 pts)

Write 3–5 sentences per answer in the markdown cell below each question.

### SA01. LoRA rank tradeoff

Explain the tradeoff when choosing LoRA rank `r`. What happens at r=1 vs r=64? Which rank does `course1/chapter2_lora/class1_decoder_lora/configs/default.yaml` use and why?

*Your answer here.*

### SA02. parents[1] Docker fix

Why does `Path(__file__).resolve().parents[4]` crash inside a Docker container built with `WORKDIR /app` and source at `/app/app/main.py`? What is the correct index and why?

*Your answer here.*

### SA03. DPO loss and reference model

Write the DPO loss in words (not LaTeX). What role does the frozen reference model play, and why is it needed?

*Your answer here.*

### SA04. EWC Fisher intuition

Explain in plain words why the Fisher information diagonal is a good proxy for 'how important is this parameter to Task A'.

*Your answer here.*

### SA05. LCEL streaming

How does LCEL streaming differ from batch `.invoke()`? Where in `01_smolsearch/app/main.py` is streaming used and what endpoint does it serve?

*Your answer here.*

### SA06. Semantic cache cosine

Why does a semantic cache use cosine similarity rather than exact string match? What threshold does the `04_llmops_baseline` stack use and what happens on a miss?

*Your answer here.*

### SA07. GCP IAM three roles

Name and justify the three IAM roles required for the GitHub Actions deployer service account in `scripts/setup_gcp_infra.sh`.

*Your answer here.*

### SA08. LangGraph cycles

Give a concrete example where a LangGraph cycle is necessary but impossible in a linear LCEL chain.

*Your answer here.*

### SA09. QLoRA double quantisation

What is double quantisation in QLoRA and what extra memory does it save?

*Your answer here.*

### SA10. release branch strategy

Why does `deploy-01-smolsearch.yml` trigger on `release` rather than `main`? What is the intended developer workflow?

*Your answer here.*

### SA11. ReAct thought/action/obs

Describe the Thought/Action/Observation cycle in the manual ReAct loop in `03_agentflow/src/graph.py`. What regex patterns are used?

*Your answer here.*

### SA12. PSI formula

Write the Population Stability Index formula and state what PSI > 0.2 typically signals.

*Your answer here.*

### SA13. bind_tools requirement

Why does `create_tool_calling_agent` require `bind_tools()` and why does `HuggingFacePipeline` not support it?

*Your answer here.*

### SA14. FAISS index contents

What does a FAISS `IndexFlatL2` store and what does `.search(q, k)` return?

*Your answer here.*

### SA15. blue-green model swap

Describe a blue/green deployment strategy for swapping a retrained model in the `course3` pipeline. What state must be transferred?

*Your answer here.*

### Part II — Sample Answers
Run the cell below to reveal sample answers for self-assessment.

In [ ]:
SA_ANSWERS = {"SA01": "Rank r controls the expressiveness of the low-rank update: r=1 is extremely parameter-efficient but can underfit complex tasks; r=64 approaches full fine-tuning expressiveness but costs 64× more trainable params. The curriculum default uses r=16, balancing task adaptation (instruction following on SmolLM2-135M) with VRAM budget. For a 4096×4096 matrix, r=16 adds 2×4096×16=131 072 params vs 16.7 M for full FT. The rule of thumb: start at r=8–16; increase if eval loss plateaus; decrease if you hit OOM.", "SA02": "Inside Docker the resolved path is `/app/app/main.py`. Its ancestors are: [0]=`/app/app`, [1]=`/app`, [2]=`/`, so index 4 raises IndexError at import time, crashing the container before uvicorn even starts. The correct index is `parents[1]` which resolves to `/app` (the service root containing `src/`, `configs/`, etc.). This is the same relative position as `parents[4]` was on the developer's machine where the file lived deeper in the monorepo tree.", "SA03": "DPO minimises the negative log-ratio: log-prob of the chosen response minus log-prob of the rejected response, each normalised by the same ratio under the frozen reference model. The reference model acts as a KL-divergence anchor — without it the policy would trivially assign probability 1 to chosen and 0 to rejected, collapsing to degenerate outputs. The reference prevents the model from drifting too far from the supervised pre-training distribution. No separate reward model is trained; the preference signal is baked directly into the loss.", "SA04": "The Fisher diagonal F_i = E[(d/d_theta_i log p(y|x))^2] measures how sensitively the likelihood of Task A data changes when parameter i is perturbed. A large F_i means small changes to that parameter strongly change Task A predictions — so it must be preserved. A small F_i means the parameter barely influences Task A outputs and can be safely modified for Task B. The EWC penalty term lambda * sum(F_i * (theta_i - theta_A_i)^2) thus acts as a soft constraint: large-F parameters pay a high cost for moving, small-F parameters move freely.", "SA05": "`.invoke()` returns the complete response after the full forward pass. `.stream()` returns a generator that yields tokens as they are produced, allowing the HTTP response to begin before generation completes. In SmolSearch, `pipeline.stream_answer()` feeds a `StreamingResponse` at the `/stream` endpoint — the client receives newline-delimited token chunks with `text/plain` media type. This matters for perceived latency: a user sees the first token in ~200ms rather than waiting 2-5s for a 128-token response.", "SA06": "Exact match fails for paraphrases ('What is LoRA?' vs 'Explain LoRA briefly') that have identical answers. Cosine similarity between sentence embeddings captures semantic equivalence regardless of phrasing. A threshold of ~0.92 balances precision (avoid returning wrong cached answers) with recall (catch near-duplicate queries). On a miss the query is forwarded to the LLM, the response is stored with its embedding for future hits. The LLMOps stack exposes a `/metrics` endpoint that reports cache hit rate, allowing operators to tune the threshold.", "SA07": "`roles/run.admin` — allows `gcloud run deploy` to create and update Cloud Run services. `roles/artifactregistry.writer` — allows `docker push` to the Artifact Registry repository where images are stored. `roles/iam.serviceAccountUser` — allows the deployer to act as the Compute Engine default service account at deploy time (Cloud Run attaches that SA to the service). Without `serviceAccountUser` the deploy command raises a permission denied error even if the other two roles are present.", "SA08": "A ReAct agent: after each tool call the agent must decide whether to call another tool or produce a final answer. This is a conditional loop — the number of iterations is unknown at graph-construction time. In LCEL a chain is a static DAG; you cannot loop back to an earlier node. LangGraph models this as a StateGraph with a `run_agent` node that transitions back to itself until the `done` flag in state is True, then routes to END. The `max_steps` guard prevents infinite loops.", "SA09": "Standard NF4 quantisation stores a quantisation constant (scale factor) per block of 64 weights in FP32 — these constants themselves consume memory (~0.5 bits/param overhead). Double quantisation quantises those constants a second time (in FP8), reducing the overhead to ~0.02 bits/param. For a 7B model this saves ~37 MB — modest in absolute terms but meaningful when fitting the model on a 24 GB GPU alongside activations and optimizer state. The implementation is controlled by `bnb_config.bnb_4bit_use_double_quant=True` in `course1/chapter3_qlora`.", "SA10": "Triggering on `main` would deploy every merged PR, including half-finished features or broken integrations. The `release` branch is an explicit production gate: a human (or automated promotion step) must merge `main` → `release` to deploy. Workflow: developers open `feat/*` PRs targeting `main`; CI runs tests; reviewer merges to `main`; after integration testing the team cuts a release by merging `main` → `release`; deploy workflows fire. This mirrors common trunk-based development with a promotion gate and makes rollback trivial (revert the merge to `release`).", "SA11": "Each iteration: (1) build a prompt including the task, available tools, and accumulated history; (2) call the LLM; (3) parse the output with `re.search(r'Final Answer[:\\s]+(.+)', raw)` — if found, set result and break; (4) else parse `Action[:\\s]+(\\w+)` and `Action Input[:\\s]+(.+)` to extract tool name and argument; (5) dispatch via `ToolRegistry.dispatch()`; (6) append `Observation: <result>\\nThought:` to history. The history string grows each step, giving the LLM full context. `max_steps` (default 5) caps the loop to prevent runaway costs.", "SA12": "PSI = sum_i[(actual_i - expected_i) * ln(actual_i / expected_i)] where i indexes bins of the feature distribution, `expected` is the reference (training) distribution and `actual` is the current production distribution. Each bin's contribution is the KL-divergence-like term. Thresholds: PSI < 0.1 = stable; 0.1–0.2 = moderate shift (monitor); > 0.2 = significant drift (trigger retraining or alert). In `course3/chapter2/drift.py` this is computed per feature at regular intervals and logged; the pipeline auto-triggers a blue/green swap when PSI exceeds the configured threshold.", "SA13": "`bind_tools()` attaches a tool schema to the LLM's generation call, instructing the model (via OpenAI function-calling protocol or equivalent) to emit structured JSON tool invocations. `HuggingFacePipeline` wraps a raw text-generation pipeline — it has no API for injecting tool schemas into the generate call and returns plain text strings. The fix in `03_agentflow` is a manual ReAct loop: we parse `Action:` / `Action Input:` patterns from free-form LLM text output and dispatch tools ourselves, bypassing the need for `bind_tools()`.", "SA14": "It stores a flat matrix of all added embedding vectors (float32, shape [n_docs, dim]) in RAM, with no compression or approximate structure (hence 'Flat'). `.search(q, k)` computes the exact L2 distance between query vector `q` and every stored vector, returning the k nearest neighbours as two numpy arrays: `distances` (shape [1,k]) and `indices` (shape [1,k]) into the stored matrix. To retrieve document text you maintain a parallel list of texts indexed by the same integer positions. FAISS does not store metadata or text — that mapping is the application's responsibility.", "SA15": "Blue = current production model serving traffic. Green = newly retrained model. Steps: (1) deploy green alongside blue on a separate port / Cloud Run revision; (2) run eval harness on green with a held-out validation set — must meet metric band; (3) shift traffic (Cloud Run traffic splitting or load-balancer weight) 10% → 50% → 100% to green; (4) monitor error rate and latency; (5) decommission blue after a soak period. State to transfer: the FAISS index (if RAG is used), the semantic cache (or flush it), active conversation session IDs. The `course3/chapter4/deployment.py` `ModelRegistry` handles version metadata."}
for k,v in SA_ANSWERS.items():
    print(f'\n── {k} ──')
    print(v)


---
## Part III — Coding Challenges (10 × 3 pts)

All cells are self-contained with mocks. Run the mock setup cell first, then each challenge cell. Assertion cells validate your implementation.

### CC01. LCEL chain with fake LLM

Build a function `build_chain(llm, template_str)` that returns a Runnable accepting `{input}` and returns the LLM's string output. Use only `FakeLLM` from the mock setup. Call it with `template_str = 'Summarise: {input}'` and verify it returns a non-empty string.

In [ ]:
def build_chain(llm, template_str):
    # Implement: format template with 'input', call llm.invoke(), return string
    pass


In [ ]:
# Assertion — do not modify
llm = FakeLLM('This is a summary.')
chain = build_chain(llm, 'Summarise: {input}')
result = chain({'input': 'LangChain is a framework.'})
assert isinstance(result, str) and len(result) > 0, 'chain must return non-empty string'
print('CC01 PASS:', result)


### CC02. Semantic cache

Implement `SemanticCache` using `FakeEmbed` and `cosine()`. Methods: `get(query) -> str|None`, `put(query, response)`. Hit when cosine ≥ 0.92.

In [ ]:
class SemanticCache:
    def __init__(self, threshold=0.92):
        self.threshold = threshold
        self._store = []  # list of (embedding, response)
        self._emb = FakeEmbed()

    def get(self, query):
        # Return cached response if cosine >= threshold, else None
        pass

    def put(self, query, response):
        # Store embedding and response
        pass


In [ ]:
# Assertion
cache = SemanticCache(threshold=0.92)
cache.put('what is LoRA', 'LoRA is low-rank adaptation')
hit = cache.get('what is LoRA')  # exact same query → should hit
miss = cache.get('what is the capital of France')  # different topic → should miss
assert hit == 'LoRA is low-rank adaptation', f'expected cache hit, got {hit}'
assert miss is None, f'expected cache miss, got {miss}'
print('CC02 PASS: hit =', hit, '| miss =', miss)


### CC03. LoRA parameter count

Write `lora_param_count(shapes, rank)` that takes a list of `(in_dim, out_dim)` tuples (one per weight matrix) and returns total trainable LoRA parameters (both A and B matrices).

In [ ]:
def lora_param_count(shapes, rank):
    # Each (in_dim, out_dim) contributes in_dim*rank + rank*out_dim params
    pass


In [ ]:
# Assertion
# SmolLM2-135M attention Q,K,V,O projections: 4 × (576, 576), rank=16
shapes = [(576, 576)] * 4
count = lora_param_count(shapes, rank=16)
expected = 4 * 2 * 576 * 16  # = 73728
assert count == expected, f'got {count}, expected {expected}'
print(f'CC03 PASS: {count:,} trainable params for 4 attention projections at rank=16')


### CC04. Lifespan seeding

Using `tempfile`, implement a mock `SmolSearchPipeline` with an `add_documents(docs)` method and write an async `lifespan` context manager that seeds 3 documents on startup. Confirm the store has 3 docs after entering the context.

In [ ]:
import asyncio
from contextlib import asynccontextmanager

SEED_DOCS = ['doc_A: LangChain', 'doc_B: FAISS', 'doc_C: LoRA']

class MockPipeline:
    def __init__(self): self.docs = []
    def add_documents(self, docs): self.docs.extend(docs)

_pipeline = None

def get_pipeline():
    global _pipeline
    if _pipeline is None: _pipeline = MockPipeline()
    return _pipeline

@asynccontextmanager
async def lifespan(app):
    # Seed SEED_DOCS into the pipeline here
    pass  # replace with your implementation
    yield


In [ ]:
# Assertion
_pipeline = None  # reset
async def _test():
    async with lifespan(None):
        p = get_pipeline()
        assert len(p.docs) == 3, f'expected 3 seeded docs, got {len(p.docs)}'
        print('CC04 PASS: pipeline seeded with', len(p.docs), 'docs')
asyncio.get_event_loop().run_until_complete(_test())


### CC05. DPO loss (simplified)

Implement `dpo_loss(log_p_chosen, log_p_rejected, log_ref_chosen, log_ref_ref)` returning a scalar. Formula: `-log(sigmoid(beta * ((log_p_chosen - log_ref_chosen) - (log_p_rejected - log_ref_ref))))`. Use `beta=0.1`.

In [ ]:
import math

def dpo_loss(log_p_chosen, log_p_rejected, log_ref_chosen, log_ref_rejected, beta=0.1):
    # Implement DPO loss
    pass


In [ ]:
# Assertion
import math
loss = dpo_loss(-1.0, -3.0, -1.5, -2.5)
# reward_diff = ((-1.0 - -1.5) - (-3.0 - -2.5)) = 0.5 - (-0.5) = 1.0
# loss = -log(sigmoid(0.1 * 1.0)) ≈ -log(0.52498) ≈ 0.6444
assert abs(loss - 0.6444) < 0.01, f'got {loss:.4f}, expected ~0.6444'
print(f'CC05 PASS: DPO loss = {loss:.4f}')


### CC06. ReAct parser

Write `parse_react(text)` returning a dict with keys `action`, `action_input`, `final_answer` (any may be None). Parse using `re`.

In [ ]:
import re

def parse_react(text):
    # Parse Action:, Action Input:, Final Answer: from text
    pass


In [ ]:
# Assertion
t1 = 'Thought: I need to search.\nAction: web_search\nAction Input: LoRA paper\n'
t2 = 'Thought: Done.\nFinal Answer: LoRA adds low-rank matrices.'
r1 = parse_react(t1)
r2 = parse_react(t2)
assert r1['action'] == 'web_search', r1
assert r1['action_input'] == 'LoRA paper', r1
assert r1['final_answer'] is None, r1
assert r2['final_answer'] == 'LoRA adds low-rank matrices.', r2
print('CC06 PASS:', r1, r2)


### CC07. Fisher diagonal (mock)

Given a list of `(param_name, grad_value)` tuples representing one-step gradients, compute the Fisher diagonal estimate `F[name] = grad**2`. Return a dict.

In [ ]:
def compute_fisher_diag(named_grads):
    # named_grads: list of (name, grad_scalar)
    pass


In [ ]:
# Assertion
grads = [('layer.0.weight', 0.5), ('layer.0.bias', -0.2), ('layer.1.weight', 1.0)]
F = compute_fisher_diag(grads)
assert abs(F['layer.0.weight'] - 0.25) < 1e-9
assert abs(F['layer.0.bias'] - 0.04) < 1e-9
assert abs(F['layer.1.weight'] - 1.0) < 1e-9
print('CC07 PASS:', F)


### CC08. EWC penalty

Given `fisher` dict, `theta_old` dict, `theta_new` dict, compute `ewc_penalty(fisher, theta_old, theta_new, lam=1000)` = `lam/2 * sum(F[k]*(new[k]-old[k])**2)`.

In [ ]:
def ewc_penalty(fisher, theta_old, theta_new, lam=1000):
    pass


In [ ]:
# Assertion
F = {'w1': 2.0, 'w2': 0.5}
old = {'w1': 1.0, 'w2': 2.0}
new = {'w1': 1.5, 'w2': 2.0}  # w2 unchanged
pen = ewc_penalty(F, old, new, lam=1000)
# 1000/2 * (2.0*(0.5**2) + 0.5*(0.0**2)) = 500 * 0.5 = 250.0
assert abs(pen - 250.0) < 1e-9, f'got {pen}'
print(f'CC08 PASS: penalty = {pen}')


### CC09. In-memory vector store search

Using `FakeVectorStore` from mock setup: add 5 documents, search with a query, assert top result is most relevant.

In [ ]:
store = FakeVectorStore()
docs = [
    'LoRA is low-rank adaptation for fine-tuning large language models',
    'FAISS enables fast nearest-neighbour search over dense vectors',
    'Cloud Run is a serverless container platform from Google',
    'DPO trains on preference pairs without a reward model',
    'EWC uses Fisher information to prevent catastrophic forgetting',
]
# Add documents and search
# Your code here
results = None  # replace with store.search(...)


In [ ]:
# Assertion
store2 = FakeVectorStore()
store2.add(docs)
results2 = store2.search('low-rank adaptation fine-tuning', k=1)
assert 'LoRA' in results2[0], f'expected LoRA doc, got: {results2}'
print('CC09 PASS: top result =', results2[0][:60])


### CC10. Cost tracker

Implement `CostTracker` with `record(model, input_tokens, output_tokens)` and `total_cost()`. Prices: SmolLM2-135M = $0.0/1K tokens (free local), GPT-3.5-turbo = $0.002/1K input + $0.002/1K output. Store in `self.log` list.

In [ ]:
PRICES = {
    'SmolLM2-135M-Instruct': {'input': 0.0, 'output': 0.0},
    'gpt-3.5-turbo': {'input': 0.002, 'output': 0.002},
}

class CostTracker:
    def __init__(self): self.log = []
    def record(self, model, input_tokens, output_tokens):
        pass
    def total_cost(self):
        pass


In [ ]:
# Assertion
ct = CostTracker()
ct.record('gpt-3.5-turbo', 500, 100)  # cost = (500+100)/1000 * 0.002 = 0.0012
ct.record('SmolLM2-135M-Instruct', 1000, 200)  # cost = 0.0
total = ct.total_cost()
assert abs(total - 0.0012) < 1e-9, f'got {total}'
assert len(ct.log) == 2
print(f'CC10 PASS: total cost = ${total:.4f}')


---
## Part IV — Case Studies (5 × 4 pts)

Write your answer in the markdown cell. Run the rubric reveal cell after to self-assess.

### CS01. Production RAG System Design (Google/Amazon ML onsite)

Design a production RAG system to answer questions over a 10,000-document internal knowledge base. The system must handle 500 QPS, return answers in < 2s p99, and cost < $0.001/query. Describe: (1) indexing pipeline, (2) retrieval strategy, (3) generation stack, (4) caching layer, (5) monitoring.

*Your answer here.*

### CS02. Debug: Cloud Run 502 on cold start

A Cloud Run service (SmolSearch) returns HTTP 502 on the first request after scale-to-zero. Subsequent requests succeed. Walk through your debugging steps and identify the most likely cause.

*Your answer here.*

### CS03. Fine-tuning strategy (Microsoft/OpenAI onsite)

A team has 50K instruction pairs and a SmolLM2-360M base model. They have one A100 40GB GPU for 4 hours. Compare and recommend: full fine-tuning vs LoRA vs QLoRA. Include: memory estimate, expected training time, and quality tradeoff.

*Your answer here.*

### CS04. RAG evaluation framework (Anthropic/OpenAI applied scientist)

You must evaluate a RAG pipeline before production launch. You have 200 query-answer-context triples (golden set). Design an evaluation framework covering retrieval quality, generation quality, and end-to-end correctness. Specify metrics, thresholds, and what to do on failure.

*Your answer here.*

### CS05. Multi-service production incident (Google SRE/Amazon)

At 14:32 UTC your LLMOps service begins returning HTTP 500 for 40% of /query requests. The semantic cache is at 0% hit rate (dropped from 55%). Model inference latency jumped from 850ms to 4200ms. Describe your incident response.

*Your answer here.*

### Part IV — Rubric Reveal
Run to reveal key points for each case study.

In [ ]:
CS_RUBRICS = {"CS01": ["Indexing: chunk docs (512 tokens, 10% overlap), embed with a fast encoder (all-MiniLM-L6-v2 ~14ms/doc), store in FAISS IndexHNSWFlat for ~100ms search at 10K docs. Persist index to GCS for warm restarts.", "Retrieval: MMR (k=6, lambda=0.7) for diversity; add BM25 hybrid re-rank for keyword-heavy queries. Use multi-query for ambiguous questions.", "Generation: local SmolLM2-360M on Cloud Run (free, 200ms/128 tokens); fall back to gpt-3.5-turbo for low-confidence answers (RAGAS faithfulness < 0.6).", "Caching: semantic cache (cosine ≥ 0.92) with Redis for frequent questions; target 40-60% hit rate to stay under cost budget.", "Monitoring: RAGAS eval on 50-query canary set per deploy; PSI on query embedding distribution; p99 latency dashboard; cache hit rate; error rate alert at > 1%."], "CS02": ["Check Cloud Run logs: look for startup crash vs timeout. 502 on cold start usually means the container took > 60s (default timeout) to become healthy.", "Most likely cause: model download at startup. Without a pre-baked model in the Docker image or a warm cache, `from_pretrained()` downloads >500MB on first start, exceeding Cloud Run's default 60s startup timeout.", "Fix options: (1) bake the model into the image (`RUN python -c 'from transformers import AutoModel; AutoModel.from_pretrained(\"...\")' during build`); (2) increase `--timeout` flag to 300s; (3) use Cloud Run min-instances=1 to avoid cold starts entirely.", "Also check: `HF_HOME=/tmp` is set (read-only filesystem); memory limit is >= 2Gi; the healthcheck endpoint `/health` returns 200 before the model is loaded (i.e., lifespan seeding is async and health is served immediately)."], "CS03": ["Full FT: SmolLM2-360M in BF16 = ~720MB weights + ~2.16GB gradients + ~5.76GB AdamW states ≈ 8.6GB total. Feasible on A100. But with 50K × 512 tokens, ~10 epochs = ~25M gradient steps — 4 hours is marginal.", "LoRA r=16: trainable params ≈ 0.5% of full. Memory ≈ 720MB base (frozen) + ~50MB adapter + ~150MB optimizer = <1GB total. Fast epochs. Quality: within 1-2% of full FT for instruction following. Recommended if iteration speed matters.", "QLoRA: base in NF4 (180MB) + BF16 adapters. Barely makes sense on A100 (already has room); better for consumer GPUs. Slight quality degradation vs LoRA at identical rank due to quantisation noise.", "Recommendation: LoRA r=16, lr=2e-4, 3 epochs, batch=8, gradient accumulation=4. Expected time: ~2.5h on A100. Eval every 500 steps. If quality insufficient after 3 epochs, increase to r=32 and re-run."], "CS04": ["Retrieval: Recall@k (k=3,5) — fraction of golden contexts that appear in top-k retrieved docs. Threshold: Recall@3 > 0.75. If below: check chunking strategy, embedding model quality, or consider hybrid BM25+dense.", "Generation — Faithfulness (RAGAS): fraction of answer claims grounded in retrieved context. Threshold > 0.85. If below: prompt is hallucinating; add 'only answer from the provided context' instruction.", "Generation — Answer Relevance (RAGAS): cosine similarity between question and generated answer embeddings. Threshold > 0.80. If below: generation model too weak or context insufficient.", "End-to-end — Exact Match / F1 on extractive answers; human preference score (1-5) on 50-sample subset. Threshold: avg preference ≥ 3.5.", "Failure protocol: gate deployment on all thresholds; for any miss, log failing query IDs, retrieve top-3 contexts, and inspect manually. Common fixes: increase k, improve chunking, switch to stronger generator for low-faithfulness queries."], "CS05": ["Triage (0-5 min): check error logs — 500s are from which layer? Likely cache miss → LLM call → LLM timeout. 0% cache hit + 5× latency suggests either (a) cache was cleared/restarted, or (b) embedding model changed (cosine scores now below threshold for all queries).", "Mitigate (5-15 min): if cache is empty (restart), it will warm up over time — not immediately actionable. If embedding model changed: roll back to previous image version via `gcloud run deploy --image <previous-sha>`. Increase Cloud Run instance count to handle backlog.", "Root cause (15-60 min): check deploy history — was there a recent deploy? Diff the images. Likely a dependency bump changed `sentence-transformers` version, changing embedding space (breaking cosine comparisons to cached embeddings from the old space). All old cache entries are now below threshold.", "Fix: version-pin `sentence-transformers` in `requirements.txt`; flush + rebuild cache after any embedding model change; add a canary check in CI that computes cosine similarity between identical queries before and after a deploy — must be > 0.999.", "Post-mortem: add embedding model version to cache key; alert on cache hit rate dropping > 20% in 5 min; add smoke test that verifies /query returns 200 with latency < 2s after each deploy."]}
for k, pts in CS_RUBRICS.items():
    print(f'\n── {k} key points ──')
    for i, pt in enumerate(pts, 1):
        print(f'  {i}. {pt}')
